# SigLIP image-level baseline (classification-only, NO U-Net decoder)

Scope: pretrained SigLIP (base-patch16-224) sebagai classifier image-level damage-presence pada split yang SAMA dgn `compare_cnn_vgg16_baselines.ipynb`.

Labels diturunkan dari mask yang sudah ada (`y=1{damage_ratio>=0.01}`); tidak ada anotasi baru, tidak ada Dice/IoU utk head ini, tidak ada ranking claim vs model segmentasi.

SigLIP diimplementasi manual di Keras 2 (TF 2.10): patch embed + transformer encoder + attention pooling head. Bobot pretrained (vision tower saja) dimuat dari `google/siglip-base-patch16-224` (HF safetensors).

Run from `backend/model/`. Writes only to `comparison_cls/`. Existing checkpoints/results untouched.

**Prasyarat:** `pip install huggingface_hub safetensors` (sudah terpasang). Bobot SigLIP (~775 MB) akan diunduh otomatis saat pertama kali run (di-cache di `~/.cache/huggingface`).

In [1]:
import os, glob, json, math
import numpy as np
import cv2
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(_g, True)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix)
from tensorflow.keras import layers, Model, Input, backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from safetensors import safe_open
from huggingface_hub import hf_hub_download

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE = (256, 256)
BATCH = 8
EPOCHS = 15
LR = 1e-4
FINE_LR = 1e-5
TAU_LABEL = 0.01
TRAINING_DIR = os.path.join('..', '..', 'training')
IMAGES_DIR = os.path.join(TRAINING_DIR, 'images')
TARGETS_DIR = os.path.join(TRAINING_DIR, 'targets')
OUTPUT_DIR = 'comparison_cls'
os.makedirs(OUTPUT_DIR, exist_ok=True)

_tf_version = tuple(int(p) for p in tf.__version__.split('.')[:2])
for _g in tf.config.list_physical_devices('GPU'):
    _det = tf.config.experimental.get_device_details(_g)
    _cap = _det.get('compute_capability')
    if _cap and _cap[0] >= 12 and _tf_version <= (2, 10):
        tf.config.set_visible_devices([], 'GPU')
        print('GPU disembunyikan (CUDA incompatible, compute capability', _cap, ') -> lanjut CPU.')
        break
print('TF:', tf.__version__, '| GPU terlihat:', tf.config.list_physical_devices('GPU'))

c:\Users\Jase.LAPTOP-UM736EL9\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.10.0). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


GPU disembunyikan (CUDA incompatible, compute capability (12, 0) ) -> lanjut CPU.
TF: 2.10.0 | GPU terlihat: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


c:\Users\Jase.LAPTOP-UM736EL9\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
post_images = sorted(glob.glob(os.path.join(IMAGES_DIR, '*_post_disaster.png')))
image_pairs, target_paths = [], []
for post_path in post_images:
    pre_path = post_path.replace('_post_disaster.png', '_pre_disaster.png')
    scene_id = os.path.basename(post_path).replace('_post_disaster.png', '')
    mask_path = os.path.join(TARGETS_DIR, f'{scene_id}_post_disaster_target.png')
    if not os.path.exists(mask_path): continue
    if not os.path.exists(pre_path): pre_path = post_path
    image_pairs.append((pre_path, post_path)); target_paths.append(mask_path)
tr_p, te_p, tr_m, te_m = train_test_split(image_pairs, target_paths, test_size=0.30, random_state=SEED)
va_p, ts_p, va_m, ts_m = train_test_split(te_p, te_m, test_size=0.50, random_state=SEED)
print(f'Total {len(image_pairs)} | train {len(tr_p)} | val {len(va_p)} | test {len(ts_p)}')

Total 2799 | train 1959 | val 420 | test 420


In [3]:
SIGLIP_SIZE = 224
def load_cls(post_path, mask_path):
    pp = os.fsdecode(post_path.numpy() if hasattr(post_path, 'numpy') else post_path)
    mp = os.fsdecode(mask_path.numpy() if hasattr(mask_path, 'numpy') else mask_path)
    img = cv2.imread(pp)
    if img is None: raise ValueError(f'Gagal membaca citra: {pp}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (SIGLIP_SIZE, SIGLIP_SIZE)).astype(np.float32) / 255.0
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if m is None: raise ValueError(f'Gagal membaca mask: {mp}')
    m = cv2.resize(m, IMG_SIZE, interpolation=cv2.INTER_NEAREST)
    return img, np.float32((m > 0).mean() >= TAU_LABEL)
def tf_load(pp, mp):
    img, lab = tf.py_function(load_cls, [pp, mp], [tf.float32, tf.float32])
    img.set_shape([SIGLIP_SIZE, SIGLIP_SIZE, 3]); lab.set_shape([])
    return img, lab
def aug(img, lab):
    img = tf.cond(tf.random.uniform(()) > 0.5, lambda: tf.image.flip_left_right(img), lambda: img)
    img = tf.cond(tf.random.uniform(()) > 0.5, lambda: tf.image.flip_up_down(img), lambda: img)
    return img, lab
def make_ds(pairs, masks, training=False):
    post = [p[1] for p in pairs]
    ds = tf.data.Dataset.from_tensor_slices((post, masks))
    if training: ds = ds.shuffle(len(post), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds, test_ds = make_ds(tr_p, tr_m, True), make_ds(va_p, va_m), make_ds(ts_p, ts_m)
print('splits ready:', len(tr_p), len(va_p), len(ts_p))

splits ready: 1959 420 420


In [4]:
import time as _t
_t0 = _t.time()
for _xb, _yb in train_ds.take(1):
    print('batch:', tuple(_xb.shape), tuple(_yb.shape),
          '| fraksi positif:', round(float(_yb.numpy().mean()), 4),
          f'| {_t.time()-_t0:.1f}s')
print('OK - pipeline mengalir, aman lanjut ke build/train.')
print('Jika CELL INI yang hang >5 mnt: masalahnya I/O data (path/corrupt), bukan GPU.')

batch: (8, 224, 224, 3) (8,) | fraksi positif: 0.375 | 0.1s
OK - pipeline mengalir, aman lanjut ke build/train.
Jika CELL INI yang hang >5 mnt: masalahnya I/O data (path/corrupt), bukan GPU.


In [5]:
SIGLIP_MODEL = 'google/siglip-base-patch16-224'
PATCH = 16
HIDDEN = 768
NUM_LAYERS = 12
NUM_HEADS = 12
INTER = 3072
NUM_PATCHES = (SIGLIP_SIZE // PATCH) ** 2

def _load_siglip_weights(model):
    path = hf_hub_download(SIGLIP_MODEL, 'model.safetensors')
    print('Loading SigLIP weights from:', path)
    weights = {}
    with safe_open(path, framework='np') as f:
        for k in f.keys():
            if k.startswith('vision_model.'):
                weights[k] = f.get_tensor(k)
    print('  loaded', len(weights), 'vision tensors')

    mapping = {}
    mapping['patch_embedding/kernel:0'] = 'vision_model.embeddings.patch_embedding.weight'
    mapping['patch_embedding/bias:0'] = 'vision_model.embeddings.patch_embedding.bias'
    mapping['pos_embed/pos_emb:0'] = 'vision_model.embeddings.position_embedding.weight'
    mapping['post_layernorm/gamma:0'] = 'vision_model.post_layernorm.weight'
    mapping['post_layernorm/beta:0'] = 'vision_model.post_layernorm.bias'
    for i in range(NUM_LAYERS):
        mapping[f'ln1_{i}/gamma:0'] = f'vision_model.encoder.layers.{i}.layer_norm1.weight'
        mapping[f'ln1_{i}/beta:0'] = f'vision_model.encoder.layers.{i}.layer_norm1.bias'
        mapping[f'ln2_{i}/gamma:0'] = f'vision_model.encoder.layers.{i}.layer_norm2.weight'
        mapping[f'ln2_{i}/beta:0'] = f'vision_model.encoder.layers.{i}.layer_norm2.bias'
        mapping[f'attn_{i}/q_kernel:0'] = f'vision_model.encoder.layers.{i}.self_attn.q_proj.weight'
        mapping[f'attn_{i}/q_bias:0'] = f'vision_model.encoder.layers.{i}.self_attn.q_proj.bias'
        mapping[f'attn_{i}/k_kernel:0'] = f'vision_model.encoder.layers.{i}.self_attn.k_proj.weight'
        mapping[f'attn_{i}/k_bias:0'] = f'vision_model.encoder.layers.{i}.self_attn.k_proj.bias'
        mapping[f'attn_{i}/v_kernel:0'] = f'vision_model.encoder.layers.{i}.self_attn.v_proj.weight'
        mapping[f'attn_{i}/v_bias:0'] = f'vision_model.encoder.layers.{i}.self_attn.v_proj.bias'
        mapping[f'attn_{i}/out_kernel:0'] = f'vision_model.encoder.layers.{i}.self_attn.out_proj.weight'
        mapping[f'attn_{i}/out_bias:0'] = f'vision_model.encoder.layers.{i}.self_attn.out_proj.bias'
        mapping[f'mlp_{i}/fc1_kernel:0'] = f'vision_model.encoder.layers.{i}.mlp.fc1.weight'
        mapping[f'mlp_{i}/fc1_bias:0'] = f'vision_model.encoder.layers.{i}.mlp.fc1.bias'
        mapping[f'mlp_{i}/fc2_kernel:0'] = f'vision_model.encoder.layers.{i}.mlp.fc2.weight'
        mapping[f'mlp_{i}/fc2_bias:0'] = f'vision_model.encoder.layers.{i}.mlp.fc2.bias'
    mapping['pool/proj_kernel:0'] = 'vision_model.head.mlp.fc1.weight'
    mapping['pool/proj_bias:0'] = 'vision_model.head.mlp.fc1.bias'
    mapping['pool/ln_gamma:0'] = 'vision_model.head.layernorm.weight'
    mapping['pool/ln_beta:0'] = 'vision_model.head.layernorm.bias'
    mapping['pool/fc1_kernel:0'] = 'vision_model.head.mlp.fc1.weight'
    mapping['pool/fc1_bias:0'] = 'vision_model.head.mlp.fc1.bias'
    mapping['pool/fc2_kernel:0'] = 'vision_model.head.mlp.fc2.weight'
    mapping['pool/fc2_bias:0'] = 'vision_model.head.mlp.fc2.bias'

    loaded, missing = 0, []
    for w in model.weights:
        if w.name in mapping:
            hf_key = mapping[w.name]
            if hf_key in weights:
                hf_val = weights[hf_key]
                if tuple(hf_val.shape) != tuple(w.shape):
                    if tuple(hf_val.shape) == tuple(reversed(w.shape)):
                        hf_val = hf_val.T
                    elif 'patch_embedding' in w.name and 'kernel' in w.name:
                        # HF: (out_ch, in_ch, kH, kW) -> Keras: (kH, kW, in_ch, out_ch)
                        hf_val = hf_val.transpose(2, 3, 1, 0)
                    else:
                        print(f'  SKIP {w.name}: shape {w.shape} vs HF {hf_val.shape}')
                        missing.append(w.name)
                        continue
                K.set_value(w, hf_val)
                loaded += 1
            else:
                missing.append(w.name)
    print(f'  assigned {loaded} weights; {len(missing)} unmapped (new head biases etc.)')
    if missing:
        print('  unmapped:', missing[:10])
    # Load position embedding into pos_embed/pos_emb:0 weight (already in mapping)
    # No special handling needed - it's loaded via the main mapping loop above

In [6]:
class _PosEmbed(layers.Layer):
    def __init__(self, num_patches, hidden, **kw):
        super().__init__(**kw)
        self.num_patches = num_patches
        self.hidden = hidden
    def build(self, input_shape):
        self.pos_emb = self.add_weight(
            shape=(self.num_patches, self.hidden),
            initializer='zeros', trainable=True, name='pos_emb')
    def call(self, x):
        return x + tf.expand_dims(tf.cast(self.pos_emb, x.dtype), 0)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_patches': self.num_patches, 'hidden': self.hidden})
        return cfg

class _MHAttn(layers.Layer):
    def __init__(self, num_heads, hidden, **kw):
        super().__init__(**kw)
        self.num_heads = num_heads
        self.hidden = hidden
        self.dim = hidden // num_heads
    def build(self, input_shape):
        self.q_proj = self.add_weight(shape=(self.hidden, self.hidden), initializer='glorot_uniform', name='q_kernel')
        self.q_bias = self.add_weight(shape=(self.hidden,), initializer='zeros', name='q_bias')
        self.k_proj = self.add_weight(shape=(self.hidden, self.hidden), initializer='glorot_uniform', name='k_kernel')
        self.k_bias = self.add_weight(shape=(self.hidden,), initializer='zeros', name='k_bias')
        self.v_proj = self.add_weight(shape=(self.hidden, self.hidden), initializer='glorot_uniform', name='v_kernel')
        self.v_bias = self.add_weight(shape=(self.hidden,), initializer='zeros', name='v_bias')
        self.out_proj = self.add_weight(shape=(self.hidden, self.hidden), initializer='glorot_uniform', name='out_kernel')
        self.out_bias = self.add_weight(shape=(self.hidden,), initializer='zeros', name='out_bias')
    def call(self, x):
        q = tf.matmul(x, self.q_proj) + self.q_bias
        k = tf.matmul(x, self.k_proj) + self.k_bias
        v = tf.matmul(x, self.v_proj) + self.v_bias
        # (batch, seq, hidden) -> (batch, seq, heads, dim) -> (batch, heads, seq, dim)
        q = tf.transpose(tf.reshape(q, (-1, NUM_PATCHES, self.num_heads, self.dim)), (0, 2, 1, 3))
        k = tf.transpose(tf.reshape(k, (-1, NUM_PATCHES, self.num_heads, self.dim)), (0, 2, 1, 3))
        v = tf.transpose(tf.reshape(v, (-1, NUM_PATCHES, self.num_heads, self.dim)), (0, 2, 1, 3))
        scale = 1.0 / math.sqrt(self.dim)
        scores = tf.matmul(q, k, transpose_b=True) * scale
        attn = tf.nn.softmax(scores, axis=-1)
        out = tf.matmul(attn, v)  # (batch, heads, seq, dim)
        out = tf.transpose(out, (0, 2, 1, 3))  # (batch, seq, heads, dim)
        out = tf.reshape(out, (-1, NUM_PATCHES, self.hidden))
        out = tf.matmul(out, self.out_proj) + self.out_bias
        return out
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_heads': self.num_heads, 'hidden': self.hidden})
        return cfg

class _MLPBlock(layers.Layer):
    def __init__(self, inter, hidden, **kw):
        super().__init__(**kw)
        self.inter = inter
        self.hidden = hidden
    def build(self, input_shape):
        self.fc1_w = self.add_weight(shape=(self.hidden, self.inter), initializer='glorot_uniform', name='fc1_kernel')
        self.fc1_b = self.add_weight(shape=(self.inter,), initializer='zeros', name='fc1_bias')
        self.fc2_w = self.add_weight(shape=(self.inter, self.hidden), initializer='glorot_uniform', name='fc2_kernel')
        self.fc2_b = self.add_weight(shape=(self.hidden,), initializer='zeros', name='fc2_bias')
    def call(self, x):
        x = tf.nn.gelu(tf.matmul(x, self.fc1_w) + self.fc1_b)
        x = tf.matmul(x, self.fc2_w) + self.fc2_b
        return x
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'inter': self.inter, 'hidden': self.hidden})
        return cfg

class _AttnPool(layers.Layer):
    def __init__(self, hidden, inter, **kw):
        super().__init__(**kw)
        self.hidden = hidden
        self.inter = inter
    def build(self, input_shape):
        self.proj_w = self.add_weight(shape=(2 * self.hidden, self.hidden), initializer='glorot_uniform', name='proj_kernel')
        self.proj_b = self.add_weight(shape=(self.hidden,), initializer='zeros', name='proj_bias')
        self.ln_gamma = self.add_weight(shape=(self.hidden,), initializer='ones', name='ln_gamma')
        self.ln_beta = self.add_weight(shape=(self.hidden,), initializer='zeros', name='ln_beta')
        self.fc1_w = self.add_weight(shape=(self.hidden, self.inter), initializer='glorot_uniform', name='fc1_kernel')
        self.fc1_b = self.add_weight(shape=(self.inter,), initializer='zeros', name='fc1_bias')
        self.fc2_w = self.add_weight(shape=(self.inter, self.hidden), initializer='glorot_uniform', name='fc2_kernel')
        self.fc2_b = self.add_weight(shape=(self.hidden,), initializer='zeros', name='fc2_bias')
    def call(self, x):
        avg = tf.reduce_mean(x, axis=1)
        mx = tf.reduce_max(x, axis=1)
        pooled = tf.concat([avg, mx], axis=-1)
        pooled = tf.matmul(pooled, self.proj_w) + self.proj_b
        mean = tf.reduce_mean(pooled, axis=-1, keepdims=True)
        var = tf.reduce_mean(tf.square(pooled - mean), axis=-1, keepdims=True)
        pooled = (pooled - mean) / tf.sqrt(var + 1e-5) * self.ln_gamma + self.ln_beta
        pooled = tf.nn.gelu(tf.matmul(pooled, self.fc1_w) + self.fc1_b)
        pooled = tf.matmul(pooled, self.fc2_w) + self.fc2_b
        return pooled
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'hidden': self.hidden, 'inter': self.inter})
        return cfg

class _Normalize(layers.Layer):
    def call(self, x):
        return (x - 0.5) / 0.5

def build_siglip_vision():
    inp = Input((SIGLIP_SIZE, SIGLIP_SIZE, 3), name='siglip_input')
    x = layers.Conv2D(HIDDEN, PATCH, strides=PATCH, padding='valid', name='patch_embedding')(inp)
    x = layers.Reshape((NUM_PATCHES, HIDDEN), name='patch_flatten')(x)
    x = _PosEmbed(NUM_PATCHES, HIDDEN, name='pos_embed')(x)
    for i in range(NUM_LAYERS):
        x_norm = layers.LayerNormalization(epsilon=1e-5, name=f'ln1_{i}')(x)
        attn_out = _MHAttn(NUM_HEADS, HIDDEN, name=f'attn_{i}')(x_norm)
        x = layers.Add(name=f'res1_{i}')([x, attn_out])
        x_norm2 = layers.LayerNormalization(epsilon=1e-5, name=f'ln2_{i}')(x)
        mlp_out = _MLPBlock(INTER, HIDDEN, name=f'mlp_{i}')(x_norm2)
        x = layers.Add(name=f'res2_{i}')([x, mlp_out])
    x = layers.LayerNormalization(epsilon=1e-5, name='post_layernorm')(x)
    pooled = _AttnPool(HIDDEN, INTER, name='pool')(x)
    return Model(inp, pooled, name='SigLIP_vision')

In [7]:
def build_siglip_classifier():
    vision = build_siglip_vision()
    inp = vision.input
    x = _Normalize(name='siglip_normalize')(inp)
    feats = vision(x)
    x = layers.Dropout(0.2, name='cls_dropout')(feats)
    out = layers.Dense(1, activation='sigmoid', name='cls_head')(x)
    return Model(inp, out, name='SigLIP_cls')
siglip_model = build_siglip_classifier()
print('SigLIP model built. Loading pretrained weights...')
_load_siglip_weights(siglip_model)
print('Weights loaded. Model summary:')
siglip_model.summary()

SigLIP model built. Loading pretrained weights...
Loading SigLIP weights from: C:\Users\Jase.LAPTOP-UM736EL9\.cache\huggingface\hub\models--google--siglip-base-patch16-224\snapshots\7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed\model.safetensors
  loaded 208 vision tensors
  SKIP pool/proj_kernel:0: shape (1536, 768) vs HF (3072, 768)
  SKIP pool/proj_bias:0: shape (768,) vs HF (3072,)
  assigned 203 weights; 2 unmapped (new head biases etc.)
  unmapped: ['pool/proj_kernel:0', 'pool/proj_bias:0']
Weights loaded. Model summary:
Model: "SigLIP_cls"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 siglip_input (InputLayer)   [(None, 224, 224, 3)]     0         
                                                                 
 siglip_normalize (_Normaliz  (None, 224, 224, 3)      0         
 e)                                                              
                                                        

In [8]:
def collect(model, ds):
    yt, yp = [], []
    for xb, yb in ds:
        yt.append(yb.numpy()); yp.append(model.predict(xb, verbose=0).ravel())
    return np.concatenate(yt), np.concatenate(yp)
def pick_threshold(model, ds, grid=np.arange(0.10, 0.91, 0.05)):
    yt, yp = collect(model, ds)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = f1_score(yt, yp >= t, zero_division=0)
        if f1 > best_f1: best_t, best_f1 = float(t), float(f1)
    return best_t
def eval_cls(model, ds, thr):
    yt, yp = collect(model, ds)
    pred = yp >= thr
    return {'threshold': thr, 'Accuracy': float(accuracy_score(yt, pred)),
        'Precision': float(precision_score(yt, pred, zero_division=0)),
        'Recall': float(recall_score(yt, pred, zero_division=0)),
        'F1': float(f1_score(yt, pred, zero_division=0)),
        'ROC_AUC': float(roc_auc_score(yt, yp)), 'PR_AUC': float(average_precision_score(yt, yp)),
        'TN_FP_FN_TP': [int(v) for v in confusion_matrix(yt, pred).ravel()]}

def train_eval(model, tag, lr=LR, freeze_backbone=True):
    if freeze_backbone:
        for layer in model.layers:
            if 'cls_' not in layer.name and 'pool_' not in layer.name:
                layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='binary_crossentropy', metrics=['accuracy'])
    ckpt = os.path.join(OUTPUT_DIR, f'{tag}_best.keras')
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[
        ModelCheckpoint(ckpt, monitor='val_loss', mode='min', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)])
    model.save(os.path.join(OUTPUT_DIR, f'{tag}_final.keras'))
    thr = pick_threshold(model, val_ds)
    row = eval_cls(model, test_ds, thr)
    row['model'] = tag
    return model, row

In [9]:
import pandas as pd
results = []
siglip_model, siglip_row = train_eval(siglip_model, 'siglip_cls', freeze_backbone=True)
results.append(siglip_row)
comp = pd.DataFrame(results).set_index('model')
comp.to_csv(os.path.join(OUTPUT_DIR, 'cls_siglip_results.csv'))
json.dump(results, open(os.path.join(OUTPUT_DIR, 'cls_siglip_results.json'), 'w'), indent=2)
comp

Epoch 1/15
245/245 [==============================] - ETA: 0s - loss: 0.7326 - accuracy: 0.5391
Epoch 1: val_loss improved from inf to 0.69508, saving model to comparison_cls\siglip_cls_best.keras
245/245 [==============================] - 350s 1s/step - loss: 0.7326 - accuracy: 0.5391 - val_loss: 0.6951 - val_accuracy: 0.5000 - lr: 1.0000e-04
Epoch 2/15
245/245 [==============================] - ETA: 0s - loss: 0.7093 - accuracy: 0.5610
Epoch 2: val_loss improved from 0.69508 to 0.64049, saving model to comparison_cls\siglip_cls_best.keras
245/245 [==============================] - 483s 2s/step - loss: 0.7093 - accuracy: 0.5610 - val_loss: 0.6405 - val_accuracy: 0.7429 - lr: 1.0000e-04
Epoch 3/15
245/245 [==============================] - ETA: 0s - loss: 0.6888 - accuracy: 0.5804
Epoch 3: val_loss improved from 0.64049 to 0.61658, saving model to comparison_cls\siglip_cls_best.keras
245/245 [==============================] - 448s 2s/step - loss: 0.6888 - accuracy: 0.5804 - val_loss: 0

,threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TN_FP_FN_TP
model,,,,,,,,
siglip_cls,0.5,0.8,0.820175,0.813043,0.816594,0.892334,0.917083,"[149, 41, 43, 187]"
